In [1]:
import pandas as pd


column_names = ['ID', 'Diagnosis'] + [f'Feature_{i}' for i in range(1, 31)]

df = pd.read_csv('/content/wdbc.data', header=None, names=column_names)

print("DataFrame loaded successfully with assigned columns:")
df.head()

DataFrame loaded successfully with assigned columns:


,ID,Diagnosis,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,...,Feature_21,Feature_22,Feature_23,Feature_24,Feature_25,Feature_26,Feature_27,Feature_28,Feature_29,Feature_30
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:
print("\nDataFrame Information:")
df.info()

print("\nFirst 5 rows of the DataFrame:")
print(df.head())

print("\nDescriptive Statistics for Numerical Features:")
print(df.describe())


DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          569 non-null    int64  
 1   Diagnosis   569 non-null    object 
 2   Feature_1   569 non-null    float64
 3   Feature_2   569 non-null    float64
 4   Feature_3   569 non-null    float64
 5   Feature_4   569 non-null    float64
 6   Feature_5   569 non-null    float64
 7   Feature_6   569 non-null    float64
 8   Feature_7   569 non-null    float64
 9   Feature_8   569 non-null    float64
 10  Feature_9   569 non-null    float64
 11  Feature_10  569 non-null    float64
 12  Feature_11  569 non-null    float64
 13  Feature_12  569 non-null    float64
 14  Feature_13  569 non-null    float64
 15  Feature_14  569 non-null    float64
 16  Feature_15  569 non-null    float64
 17  Feature_16  569 non-null    float64
 18  Feature_17  569 non-null    float64
 19  Featu

In [3]:
from sklearn.model_selection import train_test_split

df['Diagnosis'] = df['Diagnosis'].map({'M': 1, 'B': 0})

X = df.drop(columns=['ID', 'Diagnosis'])
y = df['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data preprocessing complete.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Data preprocessing complete.
X_train shape: (455, 30)
X_test shape: (114, 30)
y_train shape: (455,)
y_test shape: (114,)


In [4]:
import numpy as np

df_combined = pd.concat([X, y], axis=1)

correlation_matrix = df_combined.corr()

diagnosis_correlations = correlation_matrix['Diagnosis'].drop('Diagnosis')


correlation_threshold = 0.5
selected_features_corr = diagnosis_correlations[abs(diagnosis_correlations) >= correlation_threshold]
selected_feature_names = selected_features_corr.index.tolist()

print(f"Number of features selected based on correlation (>= {correlation_threshold}): {len(selected_feature_names)}")
print("Selected Features:", selected_feature_names)


X_train_selected = X_train[selected_feature_names]
X_test_selected = X_test[selected_feature_names]

print(f"\nX_train_selected shape: {X_train_selected.shape}")
print(f"X_test_selected shape: {X_test_selected.shape}")

Number of features selected based on correlation (>= 0.5): 15
Selected Features: ['Feature_1', 'Feature_3', 'Feature_4', 'Feature_6', 'Feature_7', 'Feature_8', 'Feature_11', 'Feature_13', 'Feature_14', 'Feature_21', 'Feature_23', 'Feature_24', 'Feature_26', 'Feature_27', 'Feature_28']

X_train_selected shape: (455, 15)
X_test_selected shape: (114, 15)


In [5]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

scaler.fit(X_train_selected)

X_train_scaled = scaler.transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("Features successfully scaled.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Features successfully scaled.
X_train_scaled shape: (455, 15)
X_test_scaled shape: (114, 15)


In [6]:
from sklearn.linear_model import LogisticRegression

log_reg_model = LogisticRegression(random_state=42, solver='liblinear') # Added solver for older sklearn versions


log_reg_model.fit(X_train_scaled, y_train)


y_train_pred = log_reg_model.predict(X_train_scaled)


y_test_pred = log_reg_model.predict(X_test_scaled)

print("Logistic Regression model trained and predictions made.")

Logistic Regression model trained and predictions made.


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)


training_error = 1 - train_accuracy
print(f"Training Error (1 - Accuracy): {training_error:.4f}")

testing_error = 1 - test_accuracy
print(f"Testing Error (1 - Accuracy): {testing_error:.4f}")

print("\n--- Test Set Evaluation ---")


accuracy = accuracy_score(y_test, y_test_pred)
print(f"Accuracy: {accuracy:.4f}")

precision = precision_score(y_test, y_test_pred)
print(f"Precision: {precision:.4f}")


recall = recall_score(y_test, y_test_pred)
print(f"Recall: {recall:.4f}")


f1 = f1_score(y_test, y_test_pred)
print(f"F1-score: {f1:.4f}")


conf_matrix = confusion_matrix(y_test, y_test_pred)
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=['Actual Negative (0)', 'Actual Positive (1)'],
    columns=['Predicted Negative (0)', 'Predicted Positive (1)']
)
print("\nConfusion Matrix:")
print(conf_matrix_df)

Training Error (1 - Accuracy): 0.0440
Testing Error (1 - Accuracy): 0.0263

--- Test Set Evaluation ---
Accuracy: 0.9737
Precision: 1.0000
Recall: 0.9286
F1-score: 0.9630

Confusion Matrix:
                     Predicted Negative (0)  Predicted Positive (1)
Actual Negative (0)                      72                       0
Actual Positive (1)                       3                      39


In [8]:
from sklearn.tree import DecisionTreeClassifier


dt_model = DecisionTreeClassifier(random_state=42)


dt_model.fit(X_train_scaled, y_train)


y_train_pred_dt = dt_model.predict(X_train_scaled)


y_test_pred_dt = dt_model.predict(X_test_scaled)

print("Decision Tree Classifier model trained and predictions made.")

Decision Tree Classifier model trained and predictions made.


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd


train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt)


training_error_dt = 1 - train_accuracy_dt
print(f"Decision Tree Training Error (1 - Accuracy): {training_error_dt:.4f}")


testing_error_dt = 1 - test_accuracy_dt
print(f"Decision Tree Testing Error (1 - Accuracy): {testing_error_dt:.4f}")

print("\n--- Decision Tree Test Set Evaluation ---")


accuracy_dt = accuracy_score(y_test, y_test_pred_dt)
print(f"Decision Tree Accuracy: {accuracy_dt:.4f}")


precision_dt = precision_score(y_test, y_test_pred_dt)
print(f"Decision Tree Precision: {precision_dt:.4f}")


recall_dt = recall_score(y_test, y_test_pred_dt)
print(f"Decision Tree Recall: {recall_dt:.4f}")


f1_dt = f1_score(y_test, y_test_pred_dt)
print(f"Decision Tree F1-score: {f1_dt:.4f}")


conf_matrix_dt = confusion_matrix(y_test, y_test_pred_dt)
conf_matrix_df_dt = pd.DataFrame(
    conf_matrix_dt,
    index=['Actual Negative (0)', 'Actual Positive (1)'],
    columns=['Predicted Negative (0)', 'Predicted Positive (1)']
)
print("\nDecision Tree Confusion Matrix:")
print(conf_matrix_df_dt)

Decision Tree Training Error (1 - Accuracy): 0.0000
Decision Tree Testing Error (1 - Accuracy): 0.0789

--- Decision Tree Test Set Evaluation ---
Decision Tree Accuracy: 0.9211
Decision Tree Precision: 0.9024
Decision Tree Recall: 0.8810
Decision Tree F1-score: 0.8916

Decision Tree Confusion Matrix:
                     Predicted Negative (0)  Predicted Positive (1)
Actual Negative (0)                      68                       4
Actual Positive (1)                       5                      37


Two ML issues relevant here

Feature Correlation: We explicitly addressed this by calculating the correlation of features with the 'Diagnosis' target variable and selecting only those features with an absolute correlation above a certain threshold (0.5). This helps in reducing dimensionality and potentially improving model performance by focusing on more relevant predictors.

Data Leakage (specifically during scaling): This was carefully handled when standardizing the features. We fitted the StandardScaler only on the training data (X_train_selected) and then used this fitted scaler to transform both the training (X_train_scaled) and testing (X_test_scaled) sets. This prevents information from the test set from inadvertently influencing the training process, which would lead to an overoptimistic evaluation of the model's performance.



Conclusion on overfitting/underfitting

The Logistic Regression model shows no signs of overfitting; in fact, its test error (0.0263) is slightly lower than its training error (0.0440). This indicates excellent generalization capabilities, suggesting the model has learned the underlying patterns effectively without memorizing the training data. Its performance on unseen data is robust and reliable.

Conversely, the Decision Tree model exhibits clear signs of overfitting. It achieved a perfect training error of 0.0000, meaning it classified every training example correctly. However, its testing error significantly increased to 0.0789. This large discrepancy indicates that the Decision Tree memorized the training data, including noise, and consequently struggles to generalize to new, unseen data, leading to a poorer performance on the test set.